In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from scipy import stats

# Loading data

In [ ]:
def read_core_output_QC(QC_file_path, cleaned_csv_path):
    '''Reads the core output QC Excel file and returns the relevant sheets as DataFrames.
    Parameters:
    QC_file_path (str): Path to the core output QC Excel file from core_analysis_v2.R.
    cleaned_csv_path (str): Path to the cleaned CSV file containing raw data from parse_plate.R.
    Returns:
    tuple: A tuple containing the extracted DataFrames for plate map, bleach corrected raw data, bleach deltas, refs, fraction table, rates table, cleaned data, and summary rates.
    '''
    # Extracting sheets from core QC output Excel file
    plate_map = pd.read_excel(QC_file_path, sheet_name='plate_map')
    bleach_corrected_raw_data = pd.read_excel(QC_file_path, sheet_name='bleach_corrected_raw')
    bleach_deltas = pd.read_excel(QC_file_path, sheet_name='bleach_deltas')
    refs = pd.read_excel(QC_file_path, sheet_name='refs')
    fraction_table = pd.read_excel(QC_file_path, sheet_name='fraction_table')
    rates_table = pd.read_excel(QC_file_path, sheet_name='rates_table')
    cleaned_data = pd.read_csv(cleaned_csv_path)

    # Mapping conditions to enzyme concentrations, given substrate concentration of 5 uM
    def mapping(condition):
        if condition == '1to1':
            return 5000
        elif condition == "1to2":
            return 2500
        elif condition == "1to20":
            return 250
        elif condition == "1to40":
            return 125
        elif condition == "1to80":
            return 62.5
        else:
            return None

    rates_table['Enz_nM'] = rates_table['Condition'].apply(mapping) # applying mapping

    # Calculating summary statistics for initial rates
    summary_rates = rates_table.groupby(['Variant', 'Condition', 'Enz_nM'])
    summary_rates = summary_rates.agg(
        mean_rate=('Initial_rate_uM_per_min', 'mean'),
        sd = ('Initial_rate_uM_per_min', 'std'),
        sem_rate=('Initial_rate_uM_per_min', 'sem'),
        n = ('Initial_rate_uM_per_min', 'count')
    ).reset_index()

    # Returning sheets as Pandas DataFrames
    return plate_map, bleach_corrected_raw_data, bleach_deltas, refs, fraction_table, rates_table, cleaned_data, summary_rates


# Loading data
file = 'SUBGVP_Esin_core_output_QC.xlsx' # Insert the correct path to your Excel file
cleaned_csv = 'cleaned_data_FRET_20feb2026.csv' # Insert the correct path to your cleaned CSV file
plate_map, bleach_corrected_raw_data, bleach_deltas, refs, fraction_table, rates_table, cleaned_data, summary_rates = read_core_output_QC(file, cleaned_csv)

# Get summary table cleaned

In [ ]:
def summary_table_cleaned(summary_rates, conditions_to_include, output_csv='summary_rates_cleaned.csv'):
    '''Cleans the summary rates table by removing rows with NaN values and ensuring correct data types.
    Parameters:
    summary_rates (pd.DataFrame): The summary rates DataFrame.
    conditions_to_include (list): List of conditions to include in the cleaned table.
    output_csv (str): Path to the output CSV file.
    Returns:
    pd.DataFrame: The cleaned summary rates DataFrame.
    '''
    # Filter the summary rates to include only specified conditions and drop rows with NaN values in key columns
    summary_rates_cleaned = summary_rates[summary_rates['Condition'].isin(conditions_to_include)]
    summary_rates_cleaned = summary_rates_cleaned.dropna(subset=['mean_rate', 'sd', 'sem_rate'])
    summary_rates_cleaned.reset_index(drop=True, inplace=True)
    summary_rates_cleaned.to_csv(output_csv, index=False) # Save the cleaned summary rates to a CSV file
    return summary_rates_cleaned

# Example usage
summary_overview = summary_table_cleaned(summary_rates, conditions_to_include=['1to1', '1to2', '1to20', '1to40', '1to80'], output_csv='tables/summary_rates_cleaned.csv')
print(summary_overview)

# Get mean initial rate barplot

In [ ]:
def mean_initial_rate_barplot(summary_rates, condition, output_path='mean_initial_barplot.png'):
    '''Creates a bar plot of mean initial rates for each variant under a specific condition.
    Parameters:
    summary_rates (pd.DataFrame): The summary rates DataFrame.
    condition (str): The specific condition to filter the data for plotting (e.g., '1to1').
    output_path (str): Path to the output image file.
    '''
    colors =  ['#00FF92', '#FF9100', '#FF4365'] # Custom colors for the bars
    condition_data = summary_rates[summary_rates['Condition'] == condition] # Filter data for the specified condition
    plt.figure(figsize=(10, 6))
    plt.bar(condition_data['Variant'], condition_data['mean_rate'], yerr=condition_data['sem_rate'], capsize=5, color=colors) # Create bar plot with error bars representing SEM
    plt.title(f'Mean Initial Rates for {condition} Condition') 
    plt.xlabel('Variant')
    plt.ylabel('Mean Initial Rate (uM/min)')
    plt.xticks(rotation=45)
    plt.grid()
    plt.savefig(output_path) # Save the plot to the specified path
    plt.show()

# For loop to create bar plots for each condition
#for condition in summary_rates['Condition'].unique():
#    mean_initial_rate_barplot(summary_rates, condition)

# Example usage
mean_initial_rate_barplot(summary_rates, '1to20', output_path='images/mean_initial_barplot_1to20.png')

In [ ]:
def mean_initial_rates(summary_rates, conditions_list, plot=True, output_plot='mean_initial_rates_all_conditions.png', output_csv='mean_initial_rates_filtered.csv'):
    '''Calculates mean initial rates for specified conditions and optionally creates a bar plot.
    Parameters:
    summary_rates (pd.DataFrame): The summary rates DataFrame.
    conditions_list (list): List of conditions to include in the analysis (e.g., ['1to1', '1to2']).
    plot (bool): Whether to create a bar plot.
    output_plot (str): Path to the output image file.
    output_csv (str): Path to the output CSV file.
    Returns:
    pd.DataFrame: The filtered summary rates DataFrame.
    '''
    summary_rates_filtered = summary_rates[summary_rates['Condition'].isin(conditions_list)] # Filter the summary rates to include only specified conditions
    
    if plot == True:
        colors = ['#00FF92', '#FF9100', '#FF4365']
        plt.figure(figsize=(10, 6))
        summary_rates_filtered.pivot(index='Condition', columns='Variant', values='mean_rate').plot(kind='bar', figsize=(10, 6), color=colors, yerr=summary_rates_filtered.pivot(index='Condition', columns='Variant', values='sem_rate'), capsize=5)
        plt.grid()
        plt.title('Mean Initial Rates by Condition and Variant')
        plt.xlabel('Condition')
        plt.ylabel('Mean Initial Rate (uM/min)')
        plt.xticks(rotation=45)
        plt.legend(title='Variant')
        plt.savefig(output_plot)
        plt.show()
    
    summary_rates_filtered.to_csv(output_csv, index=False) # Save the filtered summary rates to a CSV file
    return summary_rates_filtered

# Example usage
mean_init = mean_initial_rates(summary_rates, conditions_list=['1to1', '1to2','1to20', '1to40', '1to80'], output_plot='images/mean_initial_rates_all_conditions.png', output_csv='tables/mean_initial_rates_all_conditions.csv')
print(mean_init)

# Get catalytic efficiency

In [ ]:
def get_cat_eff(summary_rates, conditions_to_include, output_csv='catalytic_efficiency_table.csv', plot=True, output_plot='dose_response_curves.png'):
    '''Calculates catalytic efficiency (slope) for each variant based on mean initial rates across specified conditions and optionally creates dose-response curves.
    Parameters:
    summary_rates (pd.DataFrame): The summary rates DataFrame.
    conditions_to_include (list): List of conditions to include in the analysis (e.g., ['1to20', '1to40', '1to80']).
    output_csv (str): Path to the output CSV file for catalytic efficiency results.
    plot (bool): Whether to create a plot.
    output_plot (str): Path to the output image file.
    Returns:
    pd.DataFrame: Table of regression results, including the k_obs (slope) for each variant.
    '''

    sum_rates = summary_rates[summary_rates['Condition'].isin(conditions_to_include)] # Filter the summary rates to include only specified conditions
    dose_response = pd.DataFrame(columns=['Variant', 'Slope', 'Slope_SE', 'Intercept', 'Intercept_SE', 'R2', 'P_value_slope', 'n_points']) # Initialize an empty DataFrame to store regression results
    
    alternative_names = {'SubG': "P1'=G", 'SubV': "P1'=V", 'SubP': "P1'=P"}
    sum_rates['Enz_uM'] = sum_rates['Enz_nM'] * 10**(-3) # Convert nM to uM for regression analysis

    for variant in sum_rates['Variant'].unique():
        x = sum_rates[sum_rates['Variant'] == variant]['Enz_uM'] 
        y = sum_rates[sum_rates['Variant'] == variant]['mean_rate'] 

        if y.isna().any(): # If there are NaN values in the mean rates, we cannot perform regression, so we will record NaN for all regression parameters
            dose_response.loc[len(dose_response)] = [variant, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, len(x)]
        else: 
            res = stats.linregress(x, y) # Perform linear regression to get slope (k_obs) and other parameters

            dose_response.loc[len(dose_response)] = [
                variant,
                f'{res.slope:.5}', # Slope (k_obs)  
                f'{res.stderr:.7}', # Standard error of the slope
                f'{res.intercept:.5}', # Intercept
                f'{res.intercept_stderr:.7}',  # Intercept standard error 
                res.rvalue ** 2, # R-squared value
                res.pvalue, # P-value for the slope
                len(x)
            ]

    dose_response.to_csv(output_csv, index=False)

    if plot == True:
        colors = {'SubG': '#00FF92', 'SubV': '#FF4365', 'SubP': '#FF9100'}
        plt.figure(figsize=(10, 6)) 

        variants = dose_response[dose_response['Slope'].notnull()]['Variant'].unique() # Identifying variants with valid slope values (i.e., those for which regression was successful)
        collapsed = dose_response[dose_response['Slope'].isnull()]['Variant'].unique() # Identifying variants with NaN slope values (i.e., inactive substrates)

        for variant in variants: # Plotting dose-response curves for variants with activity
            variant_data = sum_rates[sum_rates['Variant'] == variant]
            x = np.arange(0, 255, 1)
            y = np.array(variant_data['mean_rate'])
            intercept = np.float64(dose_response[dose_response['Variant'] == variant]['Intercept'].values[0])
            slope = np.float64(dose_response[dose_response['Variant'] == variant]['Slope'].values[0])

            plt.scatter(variant_data['Enz_uM'], variant_data['mean_rate'], label=alternative_names[variant], color=colors[variant])
            plt.plot(x, intercept + slope * x, color=colors[variant], linestyle='--', label=f'{slope:2e}x+{intercept:.5f}')
            plt.errorbar(variant_data['Enz_uM'], variant_data['mean_rate'], yerr=variant_data['sem_rate'], fmt='o', color=colors[variant], capsize=5)
            plt.grid(True)
        
        for variant in collapsed: # Plotting horizontal line at y=0 for variants with no activity
            x = np.arange(0, 255, 1)
            plt.plot(x, 0 + 0 * x, color=colors[variant], linestyle='--', label=f'{alternative_names[variant]} (no activity)')

        plt.legend()
        plt.xlim(0,255*10**(-3))
        plt.ylim(-0.01, max(sum_rates['mean_rate']) * 1.2)
        plt.title("Observed rate constants for P1'=G, P1'=V, and P1'=P substrate variants")
        plt.xlabel('Enzyme Concentration (uM)')
        plt.ylabel('Mean Initial Rate (uM/min)')
        plt.savefig(output_plot)
        plt.show()
    return dose_response

# Example usage
conditions_to_include = ['1to20', '1to40', '1to80']
catalytic_efficiency_table = get_cat_eff(summary_rates, conditions_to_include, plot=True, output_plot='images/dose_response_curves_20to80.png', output_csv='tables/dose_response_curves_20to80.csv')
print(catalytic_efficiency_table)



# Variant well data

In [ ]:
def variant_plot(variant,
                 Intensity_v_time,
                 Fraction_raw_v_time,
                 Intensity_corr_v_time,
                 Bleach_deltas_v_time,
                 R_corr,
                 Fraction_cleaved_v_time):
    '''Creates various plots for a specific variant based on the cleaned data and QC outputs.
    Parameters:
    - variant: The variant to plot (e.g., 'SubG', 'SubV', 'SubP')
    - Intensity_v_time: File path to save the Intensity vs Time plot (raw). 
        - Donor and Acceptor intensities will be plotted together as a scatter plot across time.
    - Fraction_raw_v_time: File path to save the Fraction (raw) vs Time plot
        - Fraction (raw) will be calculated as Intensity_donor / Intensity_acceptor and plotted across time.
    - Intensity_corr_v_time: File path to save the Intensity vs Time plot (bleach corrected)
        - Donor and Acceptor intensities from the bleach corrected raw data will be plotted together as a scatter plot across time.
    - Bleach_deltas_v_time: File path to save the Bleach Deltas vs Time plot
        - delta_D and delta_A from the bleach deltas data will be plotted as scatter plots across time.
    - R_corr: File path to save the Fraction (corrected) vs Time plot
        - R_corr from the fraction table will be plotted across time.
    - Fraction_cleaved_v_time: File path to save the Fraction (normalized) vs Time plot
        - Fraction_cleaved from the fraction table will be plotted across time, with a horizontal line at y=0.3 to indicate 30% cleaved.
    '''
    

    cleaned_data['Fraction_raw'] = cleaned_data['Intensity_donor']/cleaned_data['Intensity_acceptor'] # Calculate Fraction (raw) for the cleaned data
    cleaned_data['Condition'] = cleaned_data['Well'].map(plate_map.set_index('Well')['Condition']) # Map conditions to the cleaned data using the plate map
    cleaned_data['Variant'] = cleaned_data['Well'].map(plate_map.set_index('Well')['Variant']) # Map variants to the cleaned data using the plate map

    cleaned_data_2 = cleaned_data[cleaned_data['Well'].isin(plate_map['Well'])] # Filter cleaned data to include only wells that are present in the plate map
    cleaned_data_2 = cleaned_data_2.fillna({'Condition': 'Control'}) # Fill NaN values in the Condition column with 'Control' for wells that are not mapped to any condition 

    colors = {'Donor': '#00FF92', 'Acceptor': '#FF9100', 'Fraction': '#FF4365'} 

    alternative_names = {'SubG': "P1'=G", 'SubV': "P1'=V", 'SubP': "P1'=P"}

    # Intensity vs Time (raw)
    if Intensity_v_time != None:
        fig, axes = plt.subplots(6, 3, figsize=(15, 18))

        for i, well in enumerate(cleaned_data_2[cleaned_data_2['Variant'] == variant]['Well'].unique()):
            ax = axes[i // 3, i % 3]

            substrate = cleaned_data_2[cleaned_data_2['Well']==well]
            ax.set_title(f'Well {well}, Condition {substrate["Condition"].values[0]}')
            ax.scatter(substrate['Time_donor_min'], substrate['Intensity_donor'], label='Donor', color=colors['Donor'])
            ax.scatter(substrate['Time_donor_min'], substrate['Intensity_acceptor'], label='Acceptor', color=colors['Acceptor'])
            ax.set_xlabel('Time (min)')
            ax.set_ylabel('Intensity')
            ax.legend()
            ax.grid()
        fig.suptitle(f'Intensity vs Time for {alternative_names[variant]} Variant', fontsize=16, fontweight='bold')
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.savefig(Intensity_v_time)
        plt.show()
    
    # Fraction (raw) vs Time
    if Fraction_raw_v_time != None:
        fig, axes = plt.subplots(6, 3, figsize=(15, 18))

        cleaned_data_2['Fraction_raw'] = cleaned_data_2['Intensity_donor']/cleaned_data_2['Intensity_acceptor']

        for i, well in enumerate(cleaned_data_2[cleaned_data_2['Variant'] == variant]['Well'].unique()):
            ax = axes[i // 3, i % 3]

            substrate = cleaned_data_2[cleaned_data_2['Well']==well]
            ax.set_title(f'Well {well}, Condition {substrate["Condition"].values[0]}')
            ax.scatter(substrate['Time_donor_min'], substrate['Fraction_raw'], label='Fraction (Raw)', color=colors['Fraction'])
            ax.set_xlabel('Time (min)')
            ax.set_ylabel('Fraction (Raw)')
            ax.legend()
            ax.grid()
        fig.suptitle(f'Fraction (Raw) vs Time for {alternative_names[variant]} Variant', fontsize=16, fontweight='bold')
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.savefig(Fraction_raw_v_time)
        plt.show()
    
    # Intensity vs Time (bleach corrected)
    if Intensity_corr_v_time != None:
        fig, axes = plt.subplots(5, 3, figsize=(15, 18))

        for i, well in enumerate(bleach_corrected_raw_data[bleach_corrected_raw_data['Variant'] == variant]['Well'].unique()):
            ax = axes[i // 3, i % 3]

            substrate = bleach_corrected_raw_data[bleach_corrected_raw_data['Well']==well]

            ax.set_title(f'Well {well}, Condition {substrate["Condition"].values[0]}')
            ax.scatter(substrate['Time_donor_min'], substrate['Intensity_donor'], label='Donor (Bleach Corrected)', color=colors['Donor'])
            ax.scatter(substrate['Time_donor_min'], substrate['Intensity_acceptor'], label='Acceptor (Bleach Corrected)', color=colors['Acceptor'])
            ax.set_xlabel('Time (min)')
            ax.set_ylabel('Intensity (Bleach Corrected)')
            ax.legend()
            ax.grid()
        fig.suptitle(f'Bleach Corrected Intensity vs Time for {alternative_names[variant]} Variant', fontsize=16, fontweight='bold')
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.savefig(Intensity_corr_v_time)
        plt.show()
    
    # Bleach Deltas vs Time
    if Bleach_deltas_v_time != None:
        plt.figure(figsize=(10, 5))
        substrate = bleach_deltas[bleach_deltas['Variant'] == variant]
        plt.scatter(substrate['Cycle'], substrate['delta_D'], label='delta_D', color=colors['Donor'])
        plt.scatter(substrate['Cycle'], substrate['delta_A'], label='delta_A', color=colors['Acceptor'])
        plt.xlabel('Time (min)')
        plt.ylabel('Bleach Delta')
        plt.title(f'Bleach Deltas vs Time for {alternative_names[variant]} Variant')
        plt.grid()
        plt.legend()
        plt.savefig(Bleach_deltas_v_time)
        plt.show()
    
    # Fraction (corrected) vs Time
    if R_corr != None:
        fig, axes = plt.subplots(5, 3, figsize=(15, 18))

        for i, well in enumerate(fraction_table[fraction_table['Variant'] == variant]['Well'].unique()):
            variant = plate_map[plate_map['Well'] == well]['Variant'].values[0]
            ax = axes[i // 3, i % 3]

            substrate = fraction_table[fraction_table['Well']==well]
            ax.set_title(f'Well {well}, Condition {substrate["Condition"].values[0]}')
            ax.scatter(substrate['Time_donor_min'], substrate['R_corr'], label='Fraction (corrected)', color=colors['Fraction'])
            ax.set_xlabel('Time (min)')
            ax.set_ylabel('Fraction (corrected)')
            ax.legend()
            ax.grid()
        fig.suptitle(f'Fraction (corrected)) vs Time for {alternative_names[variant]} Variant', fontsize=16, fontweight='bold')
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.savefig(R_corr)
        plt.show()
    
    # Fraction (normalized) vs Time
    if Fraction_cleaved_v_time != None:
        fig, axes = plt.subplots(5, 3, figsize=(15, 18))

        for i, well in enumerate(fraction_table[fraction_table['Variant'] == variant]['Well'].unique()):
            variant = plate_map[plate_map['Well'] == well]['Variant'].values[0]
            ax = axes[i // 3, i % 3]

            substrate = fraction_table[fraction_table['Well']==well]
            ax.set_title(f'Well {well}, Condition {substrate["Condition"].values[0]}')
            ax.scatter(substrate['Time_donor_min'], substrate['Fraction_cleaved'], label='Fraction (normalized)', color=colors['Fraction'])
            ax.set_xlabel('Time (min)')
            ax.set_ylabel('Fraction (normalized)')
            ax.grid()
            ax.axhline(y=0.3, color='blue', linestyle='--', label='30% Cleaved')
            ax.legend()
        fig.suptitle(f'Fraction (normalized) vs Time for {alternative_names[variant]} Variant', fontsize=16, fontweight='bold')
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.savefig(Fraction_cleaved_v_time)
        plt.show()
       



## Example usage

In [ ]:
variant_plot('SubG', 
             Intensity_v_time='images/SubG_Intensity_v_time.png', 
             Fraction_raw_v_time='images/SubG_Fraction_raw_v_time.png', 
             Intensity_corr_v_time='images/SubG_Intensity_corr_v_time.png', 
             Bleach_deltas_v_time='images/SubG_Bleach_deltas_v_time.png', 
             R_corr='images/SubG_R_corr_v_time.png',
             Fraction_cleaved_v_time='images/SubG_Fraction_cleaved_v_time.png')

In [ ]:
variant_plot('SubV', 
             Intensity_v_time='images/SubV_Intensity_v_time.png', 
             Fraction_raw_v_time='images/SubV_Fraction_raw_v_time.png', 
             Intensity_corr_v_time='images/SubV_Intensity_corr_v_time.png', 
             Bleach_deltas_v_time='images/SubV_Bleach_deltas_v_time.png', 
             R_corr='images/SubV_R_corr_v_time.png',
             Fraction_cleaved_v_time='images/SubV_Fraction_cleaved_v_time.png')

In [ ]:
variant_plot('SubP', 
             Intensity_v_time='images/SubP_Intensity_v_time.png', 
             Fraction_raw_v_time='images/SubP_Fraction_raw_v_time.png', 
             Intensity_corr_v_time='images/SubP_Intensity_corr_v_time.png', 
             Bleach_deltas_v_time='images/SubP_Bleach_deltas_v_time.png', 
             R_corr='images/SubP_R_corr_v_time.png',
             Fraction_cleaved_v_time='images/SubP_Fraction_cleaved_v_time.png')

## Plot control wells

In [ ]:
# Special plot in report to show substrate controls
cleaned_data['Fraction_raw'] = cleaned_data['Intensity_donor']/cleaned_data['Intensity_acceptor'] # Calculate Fraction (raw) for the cleaned data
cleaned_data['Condition'] = cleaned_data['Well'].map(plate_map.set_index('Well')['Condition']) # Map conditions to the cleaned data using the plate map
cleaned_data['Variant'] = cleaned_data['Well'].map(plate_map.set_index('Well')['Variant']) # Map variants to the cleaned data using the plate map

cleaned_data_2 = cleaned_data[cleaned_data['Well'].isin(plate_map['Well'])] # Filter cleaned data to include only wells that are present in the plate map (i.e., those that have mapped conditions and variants)
cleaned_data_2 = cleaned_data_2.fillna({'Condition': 'Control'}) # Fill NaN values in the Condition column with 'Control' for wells that are not mapped to any condition

alternative_names = {'SubG': "P1'=G", 'SubV': "P1'=V", 'SubP': "P1'=P"}

colors = {'Donor': '#00FF92', 'Acceptor': '#FF9100', 'Fraction': '#FF4365'}

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i, well in enumerate(cleaned_data_2[cleaned_data_2['Condition'] == 'Control']['Well'].unique()):
    ax = axes[i // 3, i % 3]

    substrate = cleaned_data_2[cleaned_data_2['Well']==well]
    ax.set_title(f'Well {well}, Variant {alternative_names[substrate["Variant"].values[0]]}')
    ax.scatter(substrate['Time_donor_min'], substrate['Intensity_donor'], label='Donor', color=colors['Donor'])
    ax.scatter(substrate['Time_donor_min'], substrate['Intensity_acceptor'], label='Acceptor', color=colors['Acceptor'])
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Intensity')
    ax.legend()
    ax.grid()
fig.suptitle(f'Intensity vs Time for substrate controls', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.savefig('images/Control_Intensity_v_time.png')
plt.show()

# Fraction raw across conditions for each variant

In [ ]:
def plot_fraction_raw_v_time(output_plot='fraction_raw_v_time.png'):
    '''Creates a plot of Fraction (raw) vs Time for all variants and conditions based on the cleaned data.
    Finds mean of Fraction (raw) for each variant and condition across time and plots them as scatter plots.
    Fraction (raw) is calculated as Intensity_donor / Intensity_acceptor without any corrections applied.
    Parameters:
    output_plot (str): Path to the output image file for the plot.'''

    alternative_names = {'SubG': "P1'=G", 'SubV': "P1'=V", 'SubP': "P1'=P"}
    cleaned_data['Fraction_raw'] = cleaned_data['Intensity_donor']/cleaned_data['Intensity_acceptor'] # Calculate Fraction (raw) for the cleaned data
    cleaned_data['Condition'] = cleaned_data['Well'].map(plate_map.set_index('Well')['Condition']) # Map conditions to the cleaned data using the plate map
    cleaned_data['Variant'] = cleaned_data['Well'].map(plate_map.set_index('Well')['Variant']) # Map variants to the cleaned data using the plate map

    cleaned_data_2 = cleaned_data[cleaned_data['Well'].isin(plate_map['Well'])] # Filter cleaned data to include only wells that are present in the plate map (i.e., those that have mapped conditions and variants)
    cleaned_data_2 = cleaned_data_2.fillna({'Condition': 'Control'}) # Fill NaN values in the Condition column with 'Control' for wells that are not mapped to any condition 
    plot_data = cleaned_data_2.groupby(['Variant', 'Condition', 'Time_donor_min']).agg(
        mean_fraction=('Fraction_raw', 'mean')).reset_index() # Group by Variant, Condition, and Time to calculate mean Fraction (raw) for each time point


    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    colors = ['#F79256', '#FBD1A2', '#7DCFB6', '#00B2CA', '#1D4E89', '#FF6F91']

    for i, variant in enumerate(plot_data['Variant'].unique()):
        variant_data = plot_data[plot_data['Variant'] == variant]
        for condition in variant_data['Condition'].unique():
            condition_data = variant_data[variant_data['Condition'] == condition]
            axes[i].scatter(condition_data['Time_donor_min'], condition_data['mean_fraction'], label=f'{condition}', alpha=0.5, color=colors[plot_data['Condition'].unique().tolist().index(condition)])
            axes[i].set_title(f'{alternative_names[variant]}')
            axes[i].set_xlabel('Time (min)')
            axes[i].set_ylabel('Mean fraction (not corrected)')
            axes[i].grid(True)
            axes[i].legend()
    plt.suptitle('Ratio between donor and acceptor intensities vs time for all variants and conditions', y=0.95, fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_plot) # Save the plot to the specified path
    plt.show()

# Example usage
plot_fraction_raw_v_time(output_plot='images/fraction_raw_v_time.png')